# 🎯 02 - Center and Spread

Module 06 used `.describe()` as a first look, to spot problems. Now we read it properly. Every numeric column can be summarized with two questions:

1. **Where is the middle?** (center: mean, median, mode)
2. **How spread out are the values?** (spread: range, standard deviation, IQR, percentiles)

Two columns can have the same center and completely different spreads, and a single number is never the whole story.

## ✅ Learning Goals
- Compute the mean, median, and mode with pandas methods, and explain when they disagree
- Show that the median is **robust** to outliers and the mean is not
- Compute range, variance, standard deviation, IQR, and any percentile
- Explain `ddof`, and why pandas and NumPy give different standard deviations by default
- Read every row of `.describe()`, including what `count` tells you about missing data
- See how filling missing values (Module 06) changes the statistics

In [ ]:
import numpy as np
import pandas as pd

titanic = pd.read_csv('titanic.csv')
fare = titanic['Fare']
age = titanic['Age']

## 1. Center: Mean, Median, Mode
Every pandas `Series` has these built in, so no NumPy or SciPy is needed:

| Statistic | Method | Meaning |
|---|---|---|
| Mean | `.mean()` | The sum divided by the count. The "balance point." |
| Median | `.median()` | The middle value after sorting. Half the values are below it. |
| Mode | `.mode()` | The most common value (or values). |

In [ ]:
print(f'Fare mean:   {fare.mean():.2f}')
print(f'Fare median: {fare.median():.2f}')
print('Fare mode:  ', fare.mode().tolist())

The mean fare is more than **twice** the median. That gap is a signal, and the next section shows what causes it.

`.mode()` returns a **Series**, not a single number, because a column can have a tie for most common value. Use `.mode()[0]` when you need just one.

## 2. Robustness: Who Gets Pulled by Outliers?
A statistic is **robust** if a few extreme values barely move it. Watch what happens to a small, made-up set of fares when one very expensive ticket is added:

In [ ]:
fares = pd.Series([8, 10, 12, 15, 20])
with_outlier = pd.Series([8, 10, 12, 15, 20, 500])

print('Without outlier: mean', fares.mean(), '| median', fares.median())
print('With outlier:    mean', round(with_outlier.mean(), 1), '| median', with_outlier.median())

One value moved the mean from 13 to about 94. The median barely changed (12 → 13.5), because it only cares about the **middle position**, not how far away the extreme values are.

That's what is happening in the real Titanic fares: a small number of very expensive 1st-class tickets pull the mean far above what a typical passenger paid.

| Use the… | When… |
|---|---|
| **Median** | the data are lopsided or have extreme values (incomes, prices, house sizes) |
| **Mean** | the data are roughly symmetric, or you need a total (mean × count = total) |
| **Mode** | the data are categories, or a few repeated values (most common shoe size) |

✅ **Your Turn:** Compute the mean and median of `age`. Are they close? What does that suggest about whether age has extreme values?

In [ ]:
# Your turn here

## 3. Spread: Range, Variance, Standard Deviation
- **Range** = max − min. Easy, but it depends on only the two most extreme values.
- **Variance** = the average squared distance from the mean. Its units are squared (pounds²), which is hard to interpret.
- **Standard deviation (std)** = the square root of the variance, back in the original units. Roughly: how far a typical value is from the mean.

In [ ]:
print(f'Age range: {age.max() - age.min():.2f} years ({age.min()} to {age.max()})')
print(f'Age variance: {age.var():.2f} (years squared)')
print(f'Age std: {age.std():.2f} years')

### `ddof`: Why Pandas and NumPy Disagree
Compute the standard deviation of the same column two ways:

In [ ]:
known_ages = age.dropna()

print('pandas .std():', known_ages.std())
print('np.std():     ', np.std(known_ages))

They're different, and neither is a bug. The difference is the **`ddof`** ("delta degrees of freedom") setting:

| | Default | Divides by | Use when |
|---|---|---|---|
| pandas `.std()` / `.var()` | `ddof=1` | n − 1 | your data are a **sample** of a bigger population |
| NumPy `np.std()` / `np.var()` | `ddof=0` | n | your data **are** the whole population |

Dividing by n − 1 corrects for the fact that a sample tends to look a little less spread out than the population it came from. Titanic's 891 rows are a sample of the roughly 2,200 people aboard, so pandas' default is the right one here. With large n the difference is tiny. With 5 values, it isn't.

You can always set it explicitly, and the two will match:

In [ ]:
print(known_ages.std(ddof=0))
print(np.std(known_ages, ddof=0))

## 4. Percentiles, Quartiles, and the IQR
A **percentile** answers "what value is this percentage of the data below?" The 90th percentile of fare is the price that 90% of passengers paid less than.

The **quartiles** are the 25th, 50th, and 75th percentiles (Q1, the median, and Q3). The **interquartile range (IQR)** is Q3 − Q1: the width of the **middle half** of the data. Like the median, it ignores the extremes, so it's robust.

In [ ]:
fare.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

In [ ]:
q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1

print(f'Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Fare std = {fare.std():.2f}')

The middle half of passengers paid within a band about 23 wide, but the standard deviation is about 50. The std is inflated by the same expensive tickets that pulled the mean up. **Robust center (median) goes with robust spread (IQR); mean goes with std.**

Notice the jump from the 90th percentile (about 78) to the 99th (about 250). The top 1% of fares are in a different world from everyone else. Notebook 03 takes this apart.

## 5. Reading `.describe()` Row by Row
You've run `.describe()` since Module 06. Now you know what every row means:

In [ ]:
titanic[['Age', 'Fare', 'SibSp']].describe()

| Row | Meaning | What to look for |
|---|---|---|
| `count` | Non-missing values | Lower than the number of rows? Missing data. `Age` has 714 of 891. |
| `mean` / `50%` | Mean and median | Far apart? Lopsided data or outliers (`Fare`). |
| `std` | Standard deviation | Large relative to the mean? Very spread out. |
| `min` / `max` | The extremes | Impossible values? (A negative age, a $0 fare.) |
| `25%` / `75%` | Q1 and Q3 | Their difference is the IQR. |

You can ask for different percentiles, and turn the table sideways (`.T`) when there are many columns:

In [ ]:
titanic[['Age', 'Fare']].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2)

### 🧠 Quick Check
`Fare` has a `min` of `0.00`. Is a $0 fare a data error, a placeholder (Module 06), or something real? How could you investigate? (Hint: filter for those rows and look at the other columns.)

## 6. Summarizing Categorical Columns
`.describe()` on text columns gives a different summary: count, number of distinct values, the most common value (`top`), and how often it appears (`freq`).

In [ ]:
titanic[['Sex', 'Embarked']].describe()

In [ ]:
# The proportion in each category: the categorical version of a "summary"
titanic['Embarked'].value_counts(normalize=True, dropna=False).round(3)

## 7. Missing Values Change the Statistics
Pandas statistics **skip** missing values by default (`skipna=True`). So `age.mean()` is the mean of the **714 known ages**, not all 891 passengers.

In Module 06 you learned to fill missing values. Watch what a "harmless" median fill does to the spread:

In [ ]:
age_filled = age.fillna(age.median())

comparison = pd.DataFrame({
    'original': age.describe(),
    'median-filled': age_filled.describe(),
}).round(2)
comparison

The median didn't move, but the **std shrank** and the quartiles squeezed toward the middle. You added 177 identical values right at the center, so the data look *less* spread out than they really are.

> ⚠️ Filling missing values is a decision with consequences. If you fill, say so, and compare the statistics before and after, the way the Module 06 lab compared raw and cleaned data.

---
### 📝 Summary
- Center: `.mean()`, `.median()`, `.mode()`. The median is robust to outliers, and the mean is not.
- Spread: range, `.var()`, `.std()`, `.quantile()`, and IQR = Q3 − Q1. Pair median with IQR, and mean with std.
- Pandas uses `ddof=1` (sample) and NumPy uses `ddof=0` (population). Set it explicitly when it matters.
- `.describe()`: `count` reveals missing data, and a big mean-vs-median gap reveals lopsided data.
- Statistics skip `NaN` by default, and filling missing values can quietly shrink the spread.

## 🏋️ Practice

### Practice 1 — Center of Pokémon Stats (easy)
Load `pokemon.csv` into `poke`. For `HP`, `Attack`, and `Speed`, print the mean and median. Which column has the biggest gap between them?

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here

### Practice 2 — Same Mean, Different Spread (medium)
Create two Series with **the same mean** (for example, both average to 50) but very different standard deviations. Print the mean, std, and IQR of each. In a comment, describe how a group of people with each set of scores would be different.

In [ ]:
# Your code here

### Practice 3 — Zero Fares (harder)
Filter `titanic` for rows where `Fare == 0`. How many are there? Look at their `Pclass`, `Age`, `Ticket`, and `Embarked` columns. Then compute the mean and median `Fare` with and without them. In a Markdown cell, say whether you'd treat them as placeholders, and why.

In [ ]:
# Your code here

*Your reasoning here.*

## ✏️ Your Turn
Pick one numeric column from `pokemon.csv`. In 2–3 sentences, describe it using **one center and one spread statistic that fit its shape**, and justify your choice using the gap between its mean and median.

In [ ]:
# Your turn here

*Your description here.*

## 🔥 Challenge (Optional) — Median Absolute Deviation
The **median absolute deviation (MAD)** is a robust replacement for the standard deviation: the median of the distances between each value and the median.
```python
mad = (s - s.median()).abs().median()
```
Compute the MAD and the std for `fares` and `with_outlier` from section 2. How much does each one change when the outlier is added?

In [ ]:
# Your code here

---
✅ Next up → [Shape and Outliers](03-Shape_and_Outliers.ipynb): measuring how lopsided a column is, and finding the values that don't fit.